### QS+SA SSR Only (QA Evaluation)

In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os

# Load the JSON file
file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/QA_Weights/task024_cosmosqa_answer_generation.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Extract input-output pairs from JSON
instances = data["Instances"][2500:5000]
inputs = [instance["input"] for instance in instances]
outputs = [instance["output"][0] for instance in instances]

# Split the data into train and test sets
train_inputs, test_inputs, train_outputs, test_outputs = train_test_split(
    inputs, outputs, test_size=0.2, random_state=42
)

# Convert data to Hugging Face Dataset format
train_ds = Dataset.from_dict({"input": train_inputs, "output": train_outputs})
test_ds = Dataset.from_dict({"input": test_inputs, "output": test_outputs})

# Tokenizer setup
base_model_path = "meta-llama/Meta-Llama-3-8B"  # Replace with actual model path
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Check if tokenizer has a padding token, if not, set the eos_token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

# Tokenize datasets
tokenized_train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# Convert datasets to PyTorch format
tokenized_train_ds.set_format("torch")
tokenized_test_ds.set_format("torch")

# Create DataLoaders
batch_size = 16  # Adjust as needed
train_loader = DataLoader(tokenized_train_ds, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(tokenized_test_ds, batch_size=batch_size)

# Define the model and load weights
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_sa_noEVCL2/fine-tuned-sa-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)
DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Only_SSR_Weights/Evaluation_data/predictions_SSR_QA_SA_Final_Task_QA.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")

for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:07<03:55,  7.61s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [00:13<03:21,  6.73s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(0.07634759389960193), 'rouge2': np.float64(0.017484787121627123), 'rougeL': np.float64(0.05890638133473272), 'rougeLsum': np.float64(0.06512934194307013)}


### SA

In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os




file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/QA_QG_SA_Weights/task1312_amazonreview_polarity_classification.json"

with open(file_path, "r") as f:
    data = json.load(f)

# # Extract input-output pairs from JSON
instruction="""\nInstruction: Craft one correct answer to the question given in input. Be straight to the point and do not give incorrect answers. Only output the answer without restating the question or context. Be consistent with the context and use common sense."""
final_instruction="Now, ensure to only give the answer and not restate the question or context. \nAnswer:"
# Extract input-output pairs from JSON
instances = data["Instances"][4500:5000]
test_inputs = [instruction+instance["input"]+final_instruction for instance in instances]
test_outputs = [instance["output"][0] for instance in instances]

# Split the data into train and test sets


# Convert data to Hugging Face Dataset format
test_ds = Dataset.from_dict({"input": test_inputs, "output": test_outputs})

# Tokenizer setup
base_model_path = "meta-llama/Meta-Llama-3-8B"  # Replace with actual model path
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Check if tokenizer has a padding token, if not, set the eos_token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

# Tokenize datasets
tokenized_test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# Convert datasets to PyTorch format
tokenized_test_ds.set_format("torch")


# Define the model and load weights
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_sa_noEVCL2/fine-tuned-sa-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)
DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Only_SSR_Weights/Evaluation_data/predictions_SSR_QA_SA_Final_Task_SA.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")
batch_size=16
for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:40<20:52, 40.41s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [01:19<19:41, 39.39s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(7.894736842105263e-05), 'rouge2': np.float64(0.0), 'rougeL': np.float64(8.132221734678951e-05), 'rougeLsum': np.float64(8.248077411607121e-05)}


### QA+SA+SU SSR Only(QA Evaluation)

In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os

# Load the JSON file
file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/QA_Weights/task024_cosmosqa_answer_generation.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Extract input-output pairs from JSON
instances = data["Instances"][2500:5000]
inputs = [instance["input"] for instance in instances]
outputs = [instance["output"][0] for instance in instances]

# Split the data into train and test sets
train_inputs, test_inputs, train_outputs, test_outputs = train_test_split(
    inputs, outputs, test_size=0.2, random_state=42
)

# Convert data to Hugging Face Dataset format
train_ds = Dataset.from_dict({"input": train_inputs, "output": train_outputs})
test_ds = Dataset.from_dict({"input": test_inputs, "output": test_outputs})

# Tokenizer setup
base_model_path = "meta-llama/Meta-Llama-3-8B"  # Replace with actual model path
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Check if tokenizer has a padding token, if not, set the eos_token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

# Tokenize datasets
tokenized_train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# Convert datasets to PyTorch format
tokenized_train_ds.set_format("torch")
tokenized_test_ds.set_format("torch")

# Create DataLoaders
batch_size = 16  # Adjust as needed
train_loader = DataLoader(tokenized_train_ds, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(tokenized_test_ds, batch_size=batch_size)

# Define the model and load weights
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_su_noEVCL2/fine-tuned-su-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)
DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Only_SSR_Weights/Evaluation_data/predictions_SSR_QA_SA_SU_Final_Task_QA.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")

for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:08<04:38,  9.00s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [00:15<03:41,  7.37s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(0.0744103484508565), 'rouge2': np.float64(0.01607824443081455), 'rougeL': np.float64(0.0576396556572344), 'rougeLsum': np.float64(0.06358505965463748)}


### SA

In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os




file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/QA_QG_SA_Weights/task1312_amazonreview_polarity_classification.json"

with open(file_path, "r") as f:
    data = json.load(f)

# # Extract input-output pairs from JSON
instruction="""\nInstruction: Craft one correct answer to the question given in input. Be straight to the point and do not give incorrect answers. Only output the answer without restating the question or context. Be consistent with the context and use common sense."""
final_instruction="Now, ensure to only give the answer and not restate the question or context. \nAnswer:"
# Extract input-output pairs from JSON
instances = data["Instances"][4500:5000]
test_inputs = [instruction+instance["input"]+final_instruction for instance in instances]
test_outputs = [instance["output"][0] for instance in instances]

# Split the data into train and test sets


# Convert data to Hugging Face Dataset format
test_ds = Dataset.from_dict({"input": test_inputs, "output": test_outputs})

# Tokenizer setup
base_model_path = "meta-llama/Meta-Llama-3-8B"  # Replace with actual model path
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Check if tokenizer has a padding token, if not, set the eos_token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

# Tokenize datasets
tokenized_test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# Convert datasets to PyTorch format
tokenized_test_ds.set_format("torch")


# Define the model and load weights
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_su_noEVCL2/fine-tuned-su-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)
DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Only_SSR_Weights/Evaluation_data/predictions_SSR_QA_SA_SU_Final_Task_SA.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")
batch_size=16
for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:07<03:48,  7.36s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [00:13<03:23,  6.80s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(5.394605394605395e-05), 'rouge2': np.float64(0.0), 'rougeL': np.float64(5.394605394605395e-05), 'rougeLsum': np.float64(5.394605394605395e-05)}


In [ ]:
# import torch

# torch.cuda.set_device(2)  # Set to GPU 2
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# if torch.cuda.is_available():
#     current_device = torch.cuda.current_device()  # Get the current CUDA device index
#     device_name = torch.cuda.get_device_name(current_device)  # Get the device name
#     print(f"CUDA is currently using device {current_device}: {device_name}")
# else:
#     print("CUDA is not available.")


### QA+SA+SU SSR Only (SU Evaluation)

In [1]:
import json
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from datasets import Dataset
from torch.utils.data import DataLoader
import torch
import pyro
import pyro.distributions as dist
from tqdm import tqdm
import os




file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Latest_Weights/SU_data/task511_reddit_tifu_long_text_summarization.json"

with open(file_path, "r") as f:
    data = json.load(f)

instances = data['Instances'][4500:5000]
definition=str(data['Definition'][0])
test_inputs = [str(definition+instance['input']+"'\n\nNow give your output.\nSummarize:") for instance in instances]  # Convert to string if not already
test_outputs = [str(instance['output'][0]) if instance['output'] else "" for instance in instances]  # Handle missing output


# Split the data into train and test sets


# Convert data to Hugging Face Dataset format
test_ds = Dataset.from_dict({"input": test_inputs, "output": test_outputs})

# Tokenizer setup
base_model_path = "meta-llama/Meta-Llama-3-8B"  # Replace with actual model path
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# Check if tokenizer has a padding token, if not, set the eos_token as padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenization function
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    labels = tokenizer(
        examples["output"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs

# Tokenize datasets
tokenized_test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# Convert datasets to PyTorch format
tokenized_test_ds.set_format("torch")


# Define the model and load weights
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
fine_tuned_weights_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Synthethic_Data_Generation/fine_tuned_su_noEVCL2/fine-tuned-su-lora_noEVCL2"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, fine_tuned_weights_path)
DEVICE = model.device

# Generate predictions
predictions = []
references = []

output_file_path = "/home/panshul2/cs-546/Iterative-SSR-and-EVCL-Catastrophic-Forgetting/SSR/Only_SSR_Weights/Evaluation_data/predictions_SSR_QA_SA_SU_Final_Task_SU.json"

if os.path.exists(output_file_path):
    with open(output_file_path, "r") as f:
        saved_data = json.load(f)
else:
    saved_data = {"predictions": [], "references": []}

print("Generating predictions incrementally:")

print("Generating predictions:")
batch_size=16
for i in tqdm(range(0, len(test_inputs), batch_size)):  # Loop in batches
    batch_inputs = test_inputs[i:i + batch_size]
    batch_references = test_outputs[i:i + batch_size]

    # Tokenize the inputs in a batch
    inputs_tokenized = tokenizer(batch_inputs, padding=True, truncation=True, return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        # Generate predictions using the tokenized inputs
        generated_ids = model.generate(
                input_ids=inputs_tokenized["input_ids"],
                attention_mask=inputs_tokenized["attention_mask"],
                max_new_tokens=30,  
                min_length=10,  
                no_repeat_ngram_size=2,  
                num_return_sequences=1,
                top_p=0.9,  
                temperature=0.7  
            )
        
    batch_predictions = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend(batch_predictions)
    references.extend(batch_references)

    saved_data["predictions"].extend(batch_predictions)
    saved_data["references"].extend(batch_references)

    # Save incrementally to JSON
    with open(output_file_path, "w") as json_file:
        json.dump(saved_data, json_file, indent=4)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Generating predictions incrementally:
Generating predictions:


  0%|          | 0/32 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  3%|▎         | 1/32 [00:58<30:16, 58.60s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
  6%|▋         | 2/32 [01:48<26:35, 53.19s/it]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenize

In [2]:
import evaluate
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=predictions, references=references)

# Display the results
print("\nROUGE Scores:")
print(results)


ROUGE Scores:
{'rouge1': np.float64(0.06634876433787994), 'rouge2': np.float64(0.02121246379675153), 'rougeL': np.float64(0.049241140053019286), 'rougeLsum': np.float64(0.05516093256659227)}
